In [45]:
# Import library
import pandas as pd
import numpy as np
!pip install google-play-scraper
from google_play_scraper import app, Sort, reviews_all, reviews
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [35]:

pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining
seed = 0
np.random.seed(seed)  # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt  # Matplotlib untuk visualisasi data
import seaborn as sns  # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
from sklearn.metrics import accuracy_score
 
import datetime as dt  # Manipulasi data waktu dan tanggal
import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks
 
!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia
 
from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks
 
import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stopwords) dalam berbagai bahasa.

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [36]:
# Mengambil semua ulasan dari aplikasi dengan ID 'com.mobile.legends&hl' di Google Play Store.
pd.options.mode.chained_assignment = None 
aplikasi_id = 'com.garena.game.df'
scrapreview = reviews_all(
    aplikasi_id,            # ID Aplikasi target
    lang='id',                # Bahasa ulasan: Indonesia
    country='id',             # Negara: Indonesia
    sort=Sort.NEWEST,         # Mengambil ulasan terbaru agar datanya variatif
    count=10000               # TARGET: 10.000 data
)


In [37]:
# Menyimpan ulasan dalam file CSV
df_asli = pd.DataFrame(scrapreview)

df_asli.to_csv("ulasan_aplikasi.csv", index= False)

# Menghitung jumlah baris dan kolom
jumlah_ulasan, jumlah_kolom = df_asli.shape
jumlah_ulasan, jumlah_kolom

# Menampilkan hasil
df_asli.head()
df_asli.info()

<class 'pandas.DataFrame'>
RangeIndex: 49763 entries, 0 to 49762
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              49763 non-null  str           
 1   userName              49763 non-null  str           
 2   userImage             49763 non-null  str           
 3   content               49763 non-null  str           
 4   score                 49763 non-null  int64         
 5   thumbsUpCount         49763 non-null  int64         
 6   reviewCreatedVersion  41810 non-null  str           
 7   at                    49763 non-null  datetime64[us]
 8   replyContent          1133 non-null   str           
 9   repliedAt             1133 non-null   datetime64[us]
 10  appVersion            41810 non-null  str           
dtypes: datetime64[us](2), int64(2), str(7)
memory usage: 16.9 MB


In [38]:
# Memilih hanya kolom 'content' dan 'score' untuk analisis lebih lanjut
df_select = df_asli[['content', 'score']]
df_select.info()

<class 'pandas.DataFrame'>
RangeIndex: 49763 entries, 0 to 49762
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   content  49763 non-null  str  
 1   score    49763 non-null  int64
dtypes: int64(1), str(1)
memory usage: 3.7 MB


In [39]:
df_final = df_select.dropna()
df_final = df_select.drop_duplicates()
# Menampilkan hasil
df_final.head()
df_final.info()

<class 'pandas.DataFrame'>
Index: 37377 entries, 0 to 49761
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   content  37377 non-null  str  
 1   score    37377 non-null  int64
dtypes: int64(1), str(1)
memory usage: 3.7 MB


In [40]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
 
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka
 
    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text
 
def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text
 
def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text
 
def filteringText(text): # Menghapus stopwords dalam teks
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text
 
def stemmingText(text): # Mengurangi kata ke bentuk dasarnya yang menghilangkan imbuhan awalan dan akhiran atau ke akar kata
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
 
    # Memecah teks menjadi daftar kata
    words = text.split()
 
    # Menerapkan stemming pada setiap kata dalam daftar
    stemmed_words = [stemmer.stem(word) for word in words]
 
    # Menggabungkan kata-kata yang telah distem
    stemmed_text = ' '.join(stemmed_words)
 
    return stemmed_text
 
def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [41]:
slangwords = {"@": "di", "abis": "habis", "wtb": "beli", "masi": "masih", "wts": "jual", "wtt": "tukar", "bgt": "banget", "maks": "maksimal"}
def fix_slangwords(text):
    words = text.split()
    fixed_words = []
 
    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)
 
    fixed_text = ' '.join(fixed_words)
    return fixed_text

In [42]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
df_final['text_clean'] = df_final['content'].apply(cleaningText)
 
# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
df_final['text_casefoldingText'] = df_final['text_clean'].apply(casefoldingText)
 
# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
df_final['text_slangwords'] = df_final['text_casefoldingText'].apply(fix_slangwords)
 
# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'

for res, pkg in (("tokenizers/punkt","punkt"), ("tokenizers/punkt_tab","punkt_tab")):
    try:
        nltk.data.find(res)
    except LookupError:
        nltk.download(pkg, quiet=True)

# then re-run your tokenization
df_final['text_tokenizingText'] = df_final['text_slangwords'].apply(tokenizingText)
df_final['text_tokenizingText'] = df_final['text_slangwords'].apply(tokenizingText)
 
# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
df_final['text_stopword'] = df_final['text_tokenizingText'].apply(filteringText)
 
# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
df_final['text_akhir'] = df_final['text_stopword'].apply(toSentence)

df_final.info()

<class 'pandas.DataFrame'>
Index: 37377 entries, 0 to 49761
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   content               37377 non-null  str   
 1   score                 37377 non-null  int64 
 2   text_clean            37377 non-null  str   
 3   text_casefoldingText  37377 non-null  str   
 4   text_slangwords       37377 non-null  str   
 5   text_tokenizingText   37377 non-null  object
 6   text_stopword         37377 non-null  object
 7   text_akhir            37377 non-null  str   
dtypes: int64(1), object(2), str(5)
memory usage: 15.5+ MB


*Melabeli ulasan berdasarkan score yang diberikan oleh pengulas terhadap aplikasi*

    *Kekuranggan*
    - Label merupakan skor murni yang diberikan pengulas
    - Pemberian ulasan terkadang tidak sesuai dengan score yang diberikannya

    *Kelebihan*
    - Memerlukan lebih sedikit waktu untuk melakukan iterasi pengecekan kata

In [43]:
def tentukan_label(score):
    if score <= 2: return 'Negatif'
    elif score == 3: return 'Netral'
    else: return 'Positif'

df_final['sentiment'] = df_final['score'].apply(tentukan_label)

df_final

,content,score,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_akhir,sentiment
0,sedih bangett di hp aku udahh gabisa... padaha...,2,sedih bangett di hp aku udahh gabisa padahal m...,sedih bangett di hp aku udahh gabisa padahal m...,sedih bangett di hp aku udahh gabisa padahal m...,"[sedih, bangett, di, hp, aku, udahh, gabisa, p...","[sedih, bangett, hp, udahh, gabisa, main, hp, ...",sedih bangett hp udahh gabisa main hp akuu red...,Negatif
1,"bagus, rank di warfare sehat banget naik rank ...",5,bagus rank di warfare sehat banget naik rank m...,bagus rank di warfare sehat banget naik rank m...,bagus rank di warfare sehat banget naik rank m...,"[bagus, rank, di, warfare, sehat, banget, naik...","[bagus, rank, warfare, sehat, banget, rank, me...",bagus rank warfare sehat banget rank menang ka...,Positif
2,saya sudah main game ini dan bagus di hp yang ...,5,saya sudah main game ini dan bagus di hp yang ...,saya sudah main game ini dan bagus di hp yang ...,saya sudah main game ini dan bagus di hp yang ...,"[saya, sudah, main, game, ini, dan, bagus, di,...","[main, game, bagus, hp, main, hp, itel, ultra,...",main game bagus hp main hp itel ultra glitch g...,Positif
3,terbaik pokoknya.,5,terbaik pokoknya,terbaik pokoknya,terbaik pokoknya,"[terbaik, pokoknya]","[terbaik, pokoknya]",terbaik pokoknya,Positif
4,seru banget terus grafiknya bagus banget dan seru,5,seru banget terus grafiknya bagus banget dan seru,seru banget terus grafiknya bagus banget dan seru,seru banget terus grafiknya bagus banget dan seru,"[seru, banget, terus, grafiknya, bagus, banget...","[seru, banget, grafiknya, bagus, banget, seru]",seru banget grafiknya bagus banget seru,Positif
...,...,...,...,...,...,...,...,...,...
49755,MANTAP BETULL,5,MANTAP BETULL,mantap betull,mantap betull,"[mantap, betull]","[mantap, betull]",mantap betull,Positif
49756,Ditunggu open nya.🤭🤭,5,Ditunggu open nya,ditunggu open nya,ditunggu open nya,"[ditunggu, open, nya]","[ditunggu, open]",ditunggu open,Positif
49757,GAME TERBAIK SARAN DEV GAME NYA DI RINGANKA BI...,5,GAME TERBAIK SARAN DEV GAME NYA DI RINGANKA BI...,game terbaik saran dev game nya di ringanka bi...,game terbaik saran dev game nya di ringanka bi...,"[game, terbaik, saran, dev, game, nya, di, rin...","[game, terbaik, saran, dev, game, ringanka, bi...",game terbaik saran dev game ringanka biar pemi...,Positif
49759,udh rilis aja nih 🤣 kasih bintang 5 entar aja ...,5,udh rilis aja nih kasih bintang entar aja ma...,udh rilis aja nih kasih bintang entar aja ma...,udh rilis aja nih kasih bintang entar aja maen...,"[udh, rilis, aja, nih, kasih, bintang, entar, ...","[udh, rilis, aja, nih, kasih, bintang, entar, ...",udh rilis aja nih kasih bintang entar aja maen...,Positif


*Menggunakan fitur Ekstraksi TF-IDF*

In [ ]:
# Pisahkan data menjadi fitur (tweet) dan label (sentimen)
X = df_final['text_akhir']
y = df_final['sentiment']
 
 # Bagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [ ]:

# Ekstraksi fitur dengan TF-IDF
tfidf = TfidfVectorizer(max_features=200, min_df=17, max_df=0.8 )
X_tfidf = tfidf.fit_transform(X)
 
# Konversi hasil ekstraksi fitur menjadi dataframe
features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
 
# Menampilkan hasil ekstraksi fitur
features_df
 

*Modeling Menggunakan Naive Bayes*

In [47]:
from sklearn.naive_bayes import BernoulliNB
 
# Membuat objek model Naive Bayes (Bernoulli Naive Bayes)
naive_bayes = BernoulliNB()
 
# Melatih model Naive Bayes pada data pelatihan
naive_bayes.fit(X_train.toarray(), y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_nb = naive_bayes.predict(X_train.toarray())
y_pred_test_nb = naive_bayes.predict(X_test.toarray())
 
# Evaluasi akurasi model Naive Bayes
accuracy_train_nb = accuracy_score(y_pred_train_nb, y_train)
accuracy_test_nb = accuracy_score(y_pred_test_nb, y_test)
 
# Menampilkan akurasi
print('Naive Bayes - accuracy_train:', accuracy_train_nb)
print('Naive Bayes - accuracy_test:', accuracy_test_nb)

Naive Bayes - accuracy_train: 0.7643222634694492
Naive Bayes - accuracy_test: 0.7710005350454788


*Modeling Menggunakan Random Forest*

In [48]:
from sklearn.ensemble import RandomForestClassifier
 
# Membuat objek model Random Forest
random_forest = RandomForestClassifier()
 
# Melatih model Random Forest pada data pelatihan
random_forest.fit(X_train.toarray(), y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf = random_forest.predict(X_train.toarray())
y_pred_test_rf = random_forest.predict(X_test.toarray())
 
# Evaluasi akurasi model Random Forest
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)
 
# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf)
print('Random Forest - accuracy_test:', accuracy_test_rf)

Random Forest - accuracy_train: 0.9453864419250192
Random Forest - accuracy_test: 0.7950775815944355


In [49]:
from sklearn.linear_model import LogisticRegression
 
# Membuat objek model Logistic Regression
logistic_regression = LogisticRegression()
 
# Melatih model Logistic Regression pada data pelatihan
logistic_regression.fit(X_train.toarray(), y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_lr = logistic_regression.predict(X_train.toarray())
y_pred_test_lr = logistic_regression.predict(X_test.toarray())
 
# Evaluasi akurasi model Logistic Regression pada data pelatihan
accuracy_train_lr = accuracy_score(y_pred_train_lr, y_train)
 
# Evaluasi akurasi model Logistic Regression pada data uji
accuracy_test_lr = accuracy_score(y_pred_test_lr, y_test)
 
# Menampilkan akurasi
print('Logistic Regression - accuracy_train:', accuracy_train_lr)
print('Logistic Regression - accuracy_test:', accuracy_test_lr)

Logistic Regression - accuracy_train: 0.798668940838099
Logistic Regression - accuracy_test: 0.8053772070626003


In [50]:
from sklearn.tree import DecisionTreeClassifier
 
# Membuat objek model Decision Tree
decision_tree = DecisionTreeClassifier()
 
# Melatih model Decision Tree pada data pelatihan
decision_tree.fit(X_train.toarray(), y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_dt = decision_tree.predict(X_train.toarray())
y_pred_test_dt = decision_tree.predict(X_test.toarray())
 
# Evaluasi akurasi model Decision Tree
accuracy_train_dt = accuracy_score(y_pred_train_dt, y_train)
accuracy_test_dt = accuracy_score(y_pred_test_dt, y_test)
 
# Menampilkan akurasi
print('Decision Tree - accuracy_train:', accuracy_train_dt)
print('Decision Tree - accuracy_test:', accuracy_test_dt)

Decision Tree - accuracy_train: 0.9453864419250192
Decision Tree - accuracy_test: 0.7398341359015517


In [ ]:
tfdf = logistic_regression

*Menggunakan Ekstraksi fitur*

In [ ]:
# Bagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)